# RL agent training on Kaggle (free CPU session)

Trains the PPO agent from this repository with a large budget and records the result in the
project's model registry (Supabase). The registry runs the same champion/challenger check as
the nightly job: the new version trades on the demo account only if it passes the promotion rules.

The project runs in its own Python 3.11 environment built from `uv.lock`, the same one CI uses,
because Kaggle's system Python is newer than the project supports. That lock pins a CPU build of
torch on Linux, so no GPU is needed (a small MLP policy gains little from one).

**One-time setup (free, no card):**
1. Notebook settings → *Accelerator*: **None**, *Internet*: **On**
   (needs a phone-verified Kaggle account).
2. *Add-ons → Secrets*: add `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`, enable both for this notebook.

Candles are pulled from Supabase Storage (`--sync --no-download`), so training does not
depend on the exchange API being reachable from Kaggle.

In [ ]:
REPO_URL = "https://github.com/1Andriy2/tradebot.git"
REPO_DIR = "/kaggle/working/tradebot"
TIMESTEPS = 2_000_000

In [ ]:
import shutil, subprocess, sys

# check=True stops "Run All" here if the environment cannot be built.
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--python", "3.11",
                "--extra", "ml", "--extra", "rl", "--extra", "db"], cwd=REPO_DIR, check=True)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    os.environ[name] = secrets.get_secret(name)  # never print these

In [ ]:
cmd = [f"{REPO_DIR}/.venv/bin/tradebot", "rl-train", "--sync", "--no-download",
       "--timesteps", str(TIMESTEPS), "--device", "cpu"]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if " httpx: " not in line:  # request logs contain the Supabase URL; saved notebooks can be shared
        print(line, end="")
if proc.wait() != 0:
    raise SystemExit(proc.returncode)

The last lines printed above show the new version number, whether it was promoted, and why not
if it was rejected. The dashboard's **Навчання** page shows the same information.